<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задача 4

*Вы, наверное, уже заметили, что в CLIP можно подавать текст. А может если поменять текст, результат улучшится? А какой текст может привести к улучшению? А может использовать сразу несколько текстовых промптов, а их эмбеддинги усреднить, и вот это использовать в качестве входа в CLIP? Надо проверить!*

In [ ]:
import transformers
import torch
import requests
import torchvision
from torchvision import transforms
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
from collections import Counter
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn import metrics, preprocessing
import torch.nn.functional as F
import gc
import numpy as np

In [ ]:
BATCH_SIZE = 128

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to('cuda')
model.eval()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
imagenette_data = torchvision.datasets.Imagenette('imagenette/', download=True)
data_loader = torch.utils.data.DataLoader(imagenette_data,
                                          batch_size=BATCH_SIZE,
                                          shuffle=False,
                                          num_workers=2,
                                          collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))

In [ ]:
imagenette_data.classes

In [ ]:
truth = []
image_embeddings = torch.tensor([]).to('cuda')

for batch, t in tqdm(data_loader):
  with torch.inference_mode():
    img_inputs = processor(images=batch, padding=True, return_tensors='pt').to('cuda')
    image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to('cuda')
    image_emb /= torch.norm(image_emb, 2, dim=1, keepdim=True)
    image_embeddings = torch.cat((image_embeddings, image_emb), dim=0)
    truth += t

In [ ]:
classes_for_clip = [f'Certainly an image of {i[0]}' for i in imagenette_data.classes]

txt_inputs = tokenizer(text=classes_for_clip, padding=True, return_tensors="pt").to('cuda')
text_features = model.get_text_features(**txt_inputs)
text_emb = text_features.pooler_output.detach().to('cuda')
text_embeddings = torch.div(text_emb, torch.norm(text_emb, 2, dim=1).repeat(512, 1).T)

classes_for_clip

In [ ]:
predictions = []
logits = image_embeddings @ text_embeddings.T
predicted_cat = logits.argmax(dim=1).to('cpu')
predictions = predicted_cat.tolist()

true_freqs = Counter(truth)
predicted_freqs = Counter(predictions)

fig, ax = plt.subplots(1, 2)
fig.set_figwidth(15)
fig.suptitle('Imagenette class sizes')
ax[0].bar(true_freqs.keys(), true_freqs.values())
ax[1].bar(predicted_freqs.keys(), predicted_freqs.values())

print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')
print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')
print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')
print()

### Prompt engineering thoughts

Reformulating categories just slightly changes metrics with pertubation of about 0.002. The bare category names achieve 0.988 at all metrics, while 'A picture of {classname}' approach achieves only 0.985.

It was interesting to notice that '*{classname} wonderful picture*' made model predict tench more oftenly. Howewer, due to the trend instability, we consider that to be a coincidence

### Embedding averaging

In [ ]:
classes_options = [[f'{j} {i[0]}' for i in imagenette_data.classes] for j in ('A picture of', 'An image of', 'Certainly an image of')]
mean_text_embeddings = torch.zeros((len(classes_options[0]), 10, 512)).to('cuda')

for i, class_option in enumerate(classes_options):
  txt_inputs = tokenizer(text=class_option, padding=True, return_tensors="pt").to('cuda')
  text_features = model.get_text_features(**txt_inputs)
  text_emb = text_features.pooler_output.detach().to('cuda')
  text_emb = torch.div(text_emb, torch.norm(text_emb, 2, dim=1).repeat(512, 1).T)
  mean_text_embeddings[i] = text_emb

mean_text_embeddings = mean_text_embeddings.mean(dim=0)

predictions = []
logits = image_embeddings @ mean_text_embeddings.T
predicted_cat = logits.argmax(dim=1).to('cpu')
predictions = predicted_cat.tolist()

true_freqs = Counter(truth)
predicted_freqs = Counter(predictions)

fig, ax = plt.subplots(1, 2)
fig.set_figwidth(15)
fig.suptitle('Imagenette class sizes')
ax[0].bar(true_freqs.keys(), true_freqs.values())
ax[1].bar(predicted_freqs.keys(), predicted_freqs.values())

print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')
print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')
print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')
print()